In [1]:
import os
import requests
from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth
from typing import List
from neo4j_graphrag.embeddings.base import Embedder
from neo4j_graphrag.retrievers import VectorRetriever


load_dotenv()

driver = GraphDatabase.driver(
  os.getenv('neo4j_url'),
  auth=basic_auth("neo4j", os.getenv('neo4j_name')))


class HyperClovaXEmbeddings(Embedder):
    def __init__(self):
        self.url = "https://clovastudio.stream.ntruss.com/testapp/v1/api-tools/embedding/v2"
        api_key = os.getenv('CLOVA_API_KEY')
        if not api_key:
            raise ValueError("CLOVA_API_KEY 환경 변수가 설정되지 않았습니다.")
        
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

    def embed_query(self, text: str) -> List[float]:
        data = {"text": text}
        try:
            response = requests.post(self.url, headers=self.headers, json=data)
            response.raise_for_status() 
            result = response.json()
            
            if result and "result" in result and "embedding" in result["result"]:
                return result["result"]["embedding"]
            else:
                print(f"경고: API 응답에서 임베딩을 찾을 수 없습니다. 응답: {result}")
                return []
        except requests.exceptions.RequestException as e:
            print(f"API 요청 중 오류 발생: {e}")
            return []

embedder = HyperClovaXEmbeddings()

retriever = VectorRetriever(
    driver,
    index_name='moviePlotsEmbedding',
    embedder=embedder,
    return_properties=['title', 'plot']
)


In [47]:
import os
import requests
from typing import List, Optional, Dict, Any
from neo4j_graphrag.generation import GraphRAG
from types import SimpleNamespace

class HyperClovaXLLM:
    def __init__(
        self,
        model_name: str = "HCX-003",
        api_key: Optional[str] = None,
        base_url: str = "https://clovastudio.stream.ntruss.com/testapp/v1",
        model_params: Optional[Dict[str, Any]] = None,
    ):
        self.model_name = model_name
        self.api_key = api_key or os.getenv("CLOVA_API_KEY")
        if not self.api_key:
            raise ValueError("CLOVA_API_KEY 환경변수가 필요합니다.")
        self.base_url = base_url
        self.model_params = model_params or {}

    @property
    def headers(self):
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

    def get_messages(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> List[Dict[str, str]]:
        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        if message_history:
            messages.extend(message_history)
        messages.append({"role": "user", "content": input})
        return messages

    def invoke(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> str:
        url = f"{self.base_url}/chat-completions/{self.model_name}"
        data = {
            "messages": self.get_messages(input, message_history, system_instruction),
            "maxTokens": self.model_params.get("maxTokens", 1000),
            "temperature": self.model_params.get("temperature", 0.7),
        }
        response = requests.post(url, headers=self.headers, json=data)
        response.raise_for_status()
        result = response.json()
        # 응답 구조에 따라 content 추출

        if "result" in result and "message" in result["result"]:
            return SimpleNamespace(**result["result"]["message"])
        elif "choices" in result and len(result["choices"]) > 0:
            return SimpleNamespace(**result["choices"][0]["message"])
        else:
            return str(result)


In [48]:
llm = HyperClovaXLLM(model_name='HCX-003')

In [49]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [50]:
query_text = 'What movies are sad romances?'
response = rag.search(query_text=query_text, retriever_config={'top_k': 5})
print(response.answer)

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\generation\graphrag.py:120: DeprecationWarning: The default value of 'return_context' will change from 'False' to 'True' in a future version.
  warnings.warn(
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\retrievers\vector.py:201: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


Based on the provided context, the movie "Bed of Roses" can be considered a sad romance as it is a romantic drama about a young career girl who falls in love with a shy florist, but there isn't enough information to determine if the ending is happy or sad.


In [1]:
from neo4j import GraphDatabase, basic_auth
import openai

driver = GraphDatabase.driver(
  "neo4j://52.4.166.125:7687",
  auth=basic_auth("neo4j", "stage-alkalinity-crashes"))

In [2]:
import os
import requests
from dotenv import load_dotenv
import openai
import neo4j, neo4j_genai, neo4j_graphrag

load_dotenv()

True

In [3]:
def generate_embedding(text):
    embedding = openai.embeddings.create(input = [text], model='text-embedding-ada-002').data[0].embedding
    return embedding

In [4]:
from neo4j_genai.retrievers import VectorRetriever
from neo4j_genai.embeddings.openai import OpenAIEmbeddings
embedder = OpenAIEmbeddings(model='text-embedding-ada-002')
retriever = VectorRetriever(
    driver,
    index_name='moviePlotsEmbedding',
    embedder=embedder,
    return_properties=['title', 'plot'],
)

In [5]:
query_text = 'A movie about a shooting incident.'
retriever_result = retriever.search(query_text=query_text, top_k=3)
print(retriever_result)

items=[RetrieverResultItem(content="{'title': 'City Hall', 'plot': 'The accidental shooting of a boy in New York leads to an investigation by the Deputy Mayor, and unexpectedly far-reaching consequences.'}", metadata={'score': 0.93115234375, 'nodeLabels': None, 'id': None}), RetrieverResultItem(content="{'title': 'Usual Suspects, The', 'plot': 'A sole survivor tells of the twisty events leading up to a horrific gun battle on a boat, which begin when five criminals meet at a seemingly random police lineup.'}", metadata={'score': 0.9308319091796875, 'nodeLabels': None, 'id': None}), RetrieverResultItem(content="{'title': 'Nick of Time', 'plot': 'An unimpressive, every-day man is forced into a situation where he is told to kill a politician to save his kidnapped daughter.'}", metadata={'score': 0.9137725830078125, 'nodeLabels': None, 'id': None})] metadata={'__retriever': 'VectorRetriever'}


In [6]:
from neo4j_genai.llm import OpenAILLM
from neo4j_genai.generation import GraphRAG

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [7]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [8]:
retriever.search(query_text = 'what movies are sad romances?', top_k=5).items

[RetrieverResultItem(content="{'title': 'Bed of Roses', 'plot': 'Romantic drama about a young career girl who is swept off her feet by a shy florist, who fell in love with her after one glimpse through a shadowy window.'}", metadata={'score': 0.9117584228515625, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content="{'title': 'Postman, The (Postino, Il)', 'plot': 'Simple Italian postman learns to love poetry while delivering mail to a famous poet; he uses this to woo local beauty Beatrice.'}", metadata={'score': 0.888153076171875, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content="{'title': 'Beautiful Girls', 'plot': 'A piano player at a crossroads in his life returns home to his friends and their own problems with life and love.'}", metadata={'score': 0.88623046875, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content='{\'title\': \'American President, The\', \'plot\': "Comedy-drama about a widowed U.S. president and a lobbyist who fall in love. It\'s a

In [9]:
query_text = 'what movies are sad romances?'
response = rag.search(query_text=query_text, retriever_config={'top_k':5})
print(response.answer)

The movies "Bed of Roses" and "How to Make an American Quilt" could be considered sad romances. "Bed of Roses" is a romantic drama about a young career girl and a shy florist, which often involves emotional elements. "How to Make an American Quilt" involves tales of romance and sorrow, suggesting a mix of happy and sad romantic stories.


In [10]:
from neo4j_genai.retrievers import Text2CypherRetriever

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [11]:
from neo4j import GraphDatabase
from neo4j.time import Date

def get_node_datatype(value):
    '''
    입력된 노드 Value의 데이터 타입을 반환하는 함수
    '''
    if isinstance(value, str):
        return 'STRING'
    elif isinstance(value, int):
        return 'INTEGER'
    elif isinstance(value, float):
        return 'FLOAT'
    elif isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, list):
        return f'LIST[{get_node_datatype(value[0])}]' if value else "LIST"
    elif isinstance(value, Date):
        return 'DATE'
    else:
        return 'UNKNOWN'

In [13]:
def get_schema(uri, user, password):
    '''
    Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    '''
    driver = GraphDatabase.driver(
        uri,
        auth=basic_auth(user, password)
    )

    with driver.session() as session:
        node_query = '''
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        '''
        nodes = session.run(node_query)

        rel_query = '''
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        '''
        relationships = session.run(rel_query)
        
        rel_direction_query = '''
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        '''
        rel_directions = session.run(rel_direction_query)

        schema = {'nodes': {}, 'relationships': {}, 'relations': []}

        for record in nodes:
            label = record['label']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if label not in schema['nodes']:
                schema['nodes'][label] = {}
            schema['nodes'][label][key] = inferred_type
        
        for record in relationships:
            rel_type = record['rel_type']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if rel_type not in schema['relationships']:
                schema['relationships'][rel_type] = {}
            schema['relationships'][rel_type][key] = inferred_type
        
        for record in rel_directions:
            start_label = record['start_label'][0]
            rel_type = record['rel_type']
            end_label = record['end_label'][0]
            schema['relations'].append(f'(:{start_label})-[:{rel_type}]->(:{end_label})')
        
        return schema

def format_schema(schema):
    '''
        스키마 딕셔너리를 LLM에 제공하기 위해 원하는 형태로 formatting 하는 함수
    '''
    result = []

    result.append('Node properties:')
    for label, properties in schema['nodes'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{label} {{proprs}}')
    
    result.append('Relationship properties:')
    for rel_type, properties in schema['relationships'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{rel_type} {{{props}}}')
    
    result.append('The realtionships:')
    for relation in schema['relations']:
        result.append(relation)
    
    return '\n'.join(result)

In [14]:
schema = get_schema("neo4j://52.4.166.125:7687","neo4j", "stage-alkalinity-crashes")
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
Movie {proprs}
Genre {proprs}
User {proprs}
Actor {proprs}
Person {proprs}
Director {proprs}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The realtionships:
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(:Director)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)


c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j\_sync\driver.py:542: ResourceWarning: unclosed  Neo4jDriver: <neo4j._sync.driver.Neo4jDriver object at 0x0000022F37A76750>.
  _unclosed_resource_warn(self)
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j\_sync\driver.py:547: DeprecationWarning: Relying on Driver's destructor to close the session is deprecated. Please make sure to close the session. Use it as a context (`with` statement) or make sure to call `.close()` explicitly. Future versions of the driver will not close drivers automatically.
  _deprecation_warn(


In [15]:
examples = [
    'USER INPUT: "Which actors starred in the Toy Story? QUERY: MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.title = "Toy Story" RETURN a.name',
    "USER INPUT: 'What is the average user rating for Toy Story?' QUERY: MATCH (u:User)-[r:RATED]->(m:Movie) WHERE m.title = 'Toy Story' RETURN AVG(r.rating)"
]

In [16]:
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
)

query_text = 'Which movies did Tom Hanks star in?'
search_result = retriever.search(query_text=query_text)

In [17]:
search_result.items

[RetrieverResultItem(content="<Record m.title='Punchline'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Catch Me If You Can'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Dragnet'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Saving Mr. Banks'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Bachelor Party'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Volunteers'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Man with One Red Shoe, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Splash'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Big'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Nothing in Common'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Money Pit, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Toy Story of Terror'>", metadata=None),
 RetrieverResultItem(con

In [18]:
search_result.metadata['cypher']

"MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE a.name = 'Tom Hanks' RETURN m.title"

In [19]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [20]:
query_text = 'Which movies did Tom Hanks star in?'

search_result = retriever.search(query_text=query_text)
print('==== [Text2Cyper 를 통해 자동생성한 Cypher] ====')
print(search_result.metadata['cypher'])

response = rag.search(query_text=query_text)
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cyper 를 통해 자동생성한 Cypher] ====
MATCH (a:Actor {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN m.title

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Tom Hanks starred in the following movies:

- Punchline
- Catch Me If You Can
- Dragnet
- Saving Mr. Banks
- Bachelor Party
- Volunteers
- The Man with One Red Shoe
- Splash
- Big
- Nothing in Common
- The Money Pit
- Toy Story of Terror
- Captain Phillips
- Larry Crowne
- Cloud Atlas
- Angels & Demons
- Extremely Loud and Incredibly Close
- Charlie Wilson's War
- Toy Story 3
- From the Earth to the Moon
- The Green Mile
- Saving Private Ryan
- Toy Story 2
- Apollo 13
- Toy Story
- A League of Their Own
- Forrest Gump
- Philadelphia
- Sleepless in Seattle
- Turner & Hooch
- The 'burbs
- Joe Versus the Volcano
- Bonfire of the Vanities
- The Terminal
- The Da Vinci Code
- The Polar Express
- The Ladykillers
- You've Got Mail


In [21]:
query_text = 'Recommend 10 movies from the Comedy genre'
search_result = retriever.search(query_text=query_text)
print('==== [Text2Cyper 를 통해 자동생성한 Cypher] ====')
print(search_result.metadata['cypher'])

response = rag.search(query_text=query_text)
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cyper 를 통해 자동생성한 Cypher] ====
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) 
WHERE g.name = 'Comedy' 
RETURN m.title 
LIMIT 10

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Here are 10 movies from the Comedy genre:

1. Two Night Stand
2. Stretch
3. The Boxtrolls
4. This Is Where I Leave You
5. Tusk
6. St. Vincent
7. The Rewrite
8. Big Hero 6
9. What We Do in the Shadows
10. Let's Be Cops


In [26]:
query_text = 'Recommend only the top 5 movies with the most user reviews among Romance genre movies'
search_result = retriever.search(query_text=query_text)
print('==== [Text2Cyper 를 통해 자동생성한 Cypher] ====')
print(search_result.metadata['cypher'])

response = rag.search(query_text=query_text, return_context=True)
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cyper 를 통해 자동생성한 Cypher] ====
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Romance'})<-[:RATED]-(u:User)
RETURN m.title, COUNT(u) AS reviewCount
ORDER BY reviewCount DESC
LIMIT 5

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Here are the top 5 movies with the most user reviews among Romance genre movies:

1. Forrest Gump - 341 reviews
2. American Beauty - 220 reviews
3. True Lies - 198 reviews
4. Speed - 180 reviews
5. Beauty and the Beast - 176 reviews


In [24]:
search_result

RetrieverResult(items=[RetrieverResultItem(content="<Record m.title='Forrest Gump' reviewCount=341>", metadata=None), RetrieverResultItem(content="<Record m.title='American Beauty' reviewCount=220>", metadata=None), RetrieverResultItem(content="<Record m.title='True Lies' reviewCount=198>", metadata=None), RetrieverResultItem(content="<Record m.title='Speed' reviewCount=180>", metadata=None), RetrieverResultItem(content="<Record m.title='Beauty and the Beast' reviewCount=176>", metadata=None)], metadata={'cypher': "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Romance'}) \nMATCH (u:User)-[r:RATED]->(m) \nRETURN m.title, COUNT(r) AS reviewCount \nORDER BY reviewCount DESC \nLIMIT 5", '__retriever': 'Text2CypherRetriever'})

In [27]:
query_text = 'What genre is the next five movies? 1. Forrest Gump - 341 reviews 2. American Beauty - 220 reviews 3. True Lies - 198 reviews 4. Speed - 180 reviews 5. Beauty and the Beast - 176 reviews'
search_result = retriever.search(query_text=query_text)
print('==== [Text2Cyper 를 통해 자동생성한 Cypher] ====')
print(search_result.metadata['cypher'])

response = rag.search(query_text=query_text, return_context=True)
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cyper 를 통해 자동생성한 Cypher] ====
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) 
WHERE m.title IN ['Forrest Gump', 'American Beauty', 'True Lies', 'Speed', 'Beauty and the Beast'] 
RETURN m.title, g.name

==== [생성된 Cypher를 기반으로 최종답변생성] ====
1. Forrest Gump - War, Romance, Drama, Comedy
2. American Beauty - Drama, Romance
3. True Lies - Thriller, Romance, Action, Adventure, Comedy
4. Speed - Action, Romance, Thriller
5. Beauty and the Beast - IMAX, Musical, Romance, Children, Fantasy, Animation
